### AlexNet and post-training quantization

In this notebook we train **AlexNet** (FP32) on a small computer vision dataset, then apply **post-training quantization (PTQ)** and compare metrics. We log training and validation loss and accuracy, plus **F1 score**, **sensitivity**, and **specificity**, using the **CEMI writer** so you can view runs in the workspace (`cemi gateway` and `cemi view`).

### What is quantization?

#### Background: Precision
Before diving into quantization, it’s essential to understand precision—specifically, how computers represent decimal numbers like `1.0151` or `566132.8`. Since we can conceive of infinitely precise values (like π), but only have limited space in memory, there’s a fundamental trade-off between precision (how many significant digits can be stored) and size (how many bits are used to represent the number).

In computer engineering, these values are stored as floating point numbers, governed by the IEEE 754 floating point standard. This specification defines how bits are allocated to represent the sign, exponent, and mantissa (also called the significand, which holds the meaningful digits).

Floating point formats vary by their bit-width, and each level of precision has a different rounding error margin:
* Double-precision (`fp64`) – 64 bits, max rounding error of approximately 2−52.
* Single-precision (`float32`) – 32 bits, max rounding error of approximately 2−23.
* Half-precision (`float16`) – 16 bits, max rounding error of approximately 2−10.

![alt text](https://aman.ai/primers/ai/assets/model-compression/ieee_formats.webp)

#### Comparative Summary

| Format	| Bits | Exponent Bits | Mantissa Bits | Bias | Range (Approx.) | GPU Usage
|---|---|---|---|---|---|---|
| float16 | 16 | 5 | 10 | 15 | 10−5 to 104 | Fast inference, mixed precision
| bfloat16 | 16 | 8 | 7 | 127 | 10−38 to 1038 | Training + inference, high range
| float32 | 32 | 8 | 23 | 127 | 10−45 to 1038 | Default for training
| float64 | 64 | 11 | 52 | 1023 | 10−324 to 10308 | Rare in ML, slow on GPU

This foundation in floating point formats prepares us to understand quantization—where bit-widths are reduced even further (e.g., 8-bit, 4-bit, or binary)—to achieve efficient computation with minimal loss in model performance.

#### Background: Matrix Multiplication in GPUs

Efficient matrix multiplication is at the heart of modern deep learning acceleration on GPUs. This section provides a high-level view of how matrix-matrix multiplications are implemented and optimized on GPU hardware, with special focus on tiling, Tensor Cores, and the performance implications of quantization.

Matrix multiplications, especially General Matrix Multiplications (GEMMs), are a core computational primitive in deep learning workloads. Whether in fully connected layers, convolutions (via `im2col`), or attention mechanisms, these operations are executed billions of times during training and inference. As such, optimizing GEMM performance is essential for efficient neural network execution, particularly on GPUs.

To execute GEMMs efficiently, GPUs partition the output matrix into tiles. Each tile corresponds to a submatrix of the result and is computed by a thread block. The GPU steps through the input matrices along the shared dimension (K) in tiles, performing multiply-accumulate operations and writing the results into the corresponding tile of the output matrix. The illustration below (source) shows the tiled outer product approach to GEMMs.

![alt text](https://aman.ai/primers/ai/assets/model-compression/tiled_outer_product_gemm_placeholder.jpg)

Thread blocks are mapped to the GPU’s streaming multiprocessors (SMs), the fundamental compute units that execute instructions in parallel. Each SM can process one or more thread blocks concurrently, depending on the available resources and occupancy.

Performance in GPU matrix multiplication is often bounded by one of two factors:
* Compute (math) bound: When the arithmetic intensity (FLOPs per byte) is high enough that math operations dominate runtime.
* Memory bound: When the operation requires frequent memory access compared to math operations, limiting throughput.

Whether a given GEMM is compute- or memory-bound depends on the matrix dimensions (*M*,*N*,*K*) and the hardware’s characteristics. For example, matrix-vector products (where either *M* = 1 or *N* = 1) are typically memory-bound due to their low arithmetic intensity.

Modern NVIDIA GPUs include specialized hardware units called Tensor Cores, which are designed to accelerate GEMMs involving low-precision data types such as `float16`, `bfloat16`, and `int8`. Tensor Cores perform small matrix multiplications in parallel and require that the matrices’ dimensions align with certain multiples (e.g., 8 for `float16`, 16 for `int8`) to achieve peak performance. For instance, on Ampere and newer architectures like Hopper or Blackwell, aligning dimensions to larger multiples (e.g., 64 or 128 elements) often yields even better throughput.

Matrix dimensions that are not aligned to tile sizes lead to tile quantization, where some tiles carry less useful work, reducing efficiency. Similarly, if the total number of tiles is not an even multiple of the number of GPU SMs, wave quantization can cause underutilization. Both effects can significantly degrade performance despite identical algorithmic complexity.

To address this, libraries like cuBLAS employ heuristics or benchmarking to select optimal tile sizes, balancing between tile reuse (large tiles) and parallelism (many small tiles). Larger tiles tend to be more efficient due to better data reuse, but may reduce parallel occupancy on smaller problems.

#### Definition

***Quantization in the context of deep learning refers to the process of reducing the numerical precision of a model’s parameters (weights) and/or intermediate computations (activations).***

![alt text](https://miro.medium.com/v2/resize:fit:1204/format:webp/0*TH5XhREs4YRvAB0T)

Typically, models are trained and stored using 32-bit floating-point (`float32`) precision. Quantization replaces these high-precision values with lower-precision representations—such as 16-bit floating point (`float16`), 8-bit integer (`int8`), 4-bit integer, or binary formats in more extreme scenarios. The primary goals are to reduce model size, improve memory and compute efficiency, and accelerate inference—particularly on hardware that supports low-precision arithmetic.

For example, in symmetric quantization, the range of the original floating-point values is mapped to a symmetric range around zero in the quantized space. In the previous examples, notice how the ranges before and after quantization remain centered around zero. This means that the quantized value for zero in the floating-point space is exactly zero in the quantized space.

![alt text](https://miro.medium.com/v2/resize:fit:1204/format:webp/0*zhx02tv2blPF0WH0)

A nice example of a form of symmetric quantization is called absolute maximum (absmax) quantization. Given a list of values, we take the highest absolute value (α) as the range to perform the linear mapping.

![alt text](https://miro.medium.com/v2/resize:fit:1204/format:webp/0*ojckTi6R7MznJ1t5)

Since it is a linear mapping centered around zero, the formula is straightforward. We first calculate a scale factor (s) using:

* b is the number of bytes that we want to quantize to (8),
* α is the highest absolute value,

Then, we use the s to quantize the input x To retrieve the original FP32 values, we can use the previously calculated scaling factor (s) to dequantize the quantized values.You can see certain values, such as 3.08 and 3.02 being assigned to the INT8, namely 36. When you dequantize the values to return to FP32, they lose some precision and are not distinguishable anymore.
This is often referred to as the quantization error which we can calculate by finding the difference between the original and dequantized values.
Generally, the lower the number of bits, the more quantization error we tend to have.

*Note the [-127, 127] range of values represents the restricted range. The unrestricted range is [-128, 127] and depends on the quantization method.*

The primary goal of quantization is to enhance inference speed. In contrast, as will be discussed in the section on Mixed Precision Training, the goal of Automatic Mixed Precision (AMP) is to reduce training time. Quantization is effective, in part, because modern neural networks are typically highly over-parameterized and exhibit robustness to minor numerical perturbations. With appropriate calibration and suitable tools, lower-precision representations can approximate the full-precision model closely enough for practical deployment.


### What is Post-Training Quantization?

Post-Training Quantization (PTQ) is a technique in PyTorch that enables the conversion of a model’s weights and activations from floating-point (typically `float32`) to 8-bit integers (`int8`), significantly improving inference efficiency in terms of speed and memory usage. This method is particularly well-suited for deployment scenarios on both server and edge devices, where latency and resource constraints are critical.

To facilitate this process, PyTorch inserts special modules known as observers into the model. These modules capture the activation ranges at various points in the network. Once sufficient data has been passed through the model during calibration, the observers record min-max values or histograms (depending on the observer type), which are then used during quantization.

A key benefit of static quantization is that it allows quantized values to be passed between operations directly, eliminating the need for costly float-to-int and int-to-float conversions at each layer. This optimization significantly reduces runtime overhead and enables end-to-end execution in `int8`.

PyTorch also supports several advanced features to further improve the effectiveness of static quantization:
**Observers:**
* Observer modules are used to collect statistics on activations and weights during calibration. These can be customized to suit different data distributions or quantization strategies. PyTorch provides default observers like `MinMaxObserver` and `HistogramObserver`, and users can register them via the model’s `qconfig`.
* Observers are inserted using `torch.quantization.prepare`.

**Operator Fusion:**
* PyTorch supports the fusion of multiple operations (e.g., convolution + batch normalization + ReLU) into a single fused operator. This reduces memory access overhead and improves both runtime performance and numerical stability.
* Modules can be fused using `torch.quantization.fuse_modules`.

**Per-Channel Weight Quantization:**
* Instead of applying the same quantization parameters across all weights in a layer, per-channel quantization independently quantizes each output channel (particularly in convolution or linear layers). This approach improves accuracy while maintaining the performance benefits of quantization.
* Final conversion to the quantized model is done using `torch.quantization.convert`.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, recall_score
import io
import time
from pathlib import Path

%matplotlib inline

### Dataset

We use **CIFAR-10**: 60k small images in 10 classes. AlexNet expects 224×224 inputs, so we resize in the transform. We split into train and validation (e.g. 80/20 or a fixed val size).

In [ ]:
from torch.utils.data import DataLoader, random_split, Subset

batch_size = 32
val_size = 200
num_classes = 10
torch.manual_seed(1)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

full_dataset = datasets.CIFAR10(
    root="data/",
    download=True,
    transform=train_transform,
    train=True
)

dataset = Subset(full_dataset, range(1000))   # only use first 100 images
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size * 2, num_workers=0)

### Normalization

We use ImageNet mean and std so that pretrained AlexNet (if used) sees inputs in the range it was trained on. The same normalization is applied at inference and for PTQ calibration.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
# Rebuild datasets with full transform (resize -> totensor -> normalize)
full_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])
full_dataset = datasets.CIFAR10(
    root="data/",
    download=True,
    transform=train_transform,
    train=True
)

dataset = Subset(full_dataset, range(1000))   # only use first 100 images
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size * 2, num_workers=0)

### Evaluation metrics: accuracy, F1, sensitivity, specificity

- **Accuracy**: fraction of correct predictions.
- **F1 (macro)**: macro-averaged F1 over classes (2 × precision × recall / (precision + recall)); balances precision and recall.
- **Sensitivity** (recall): TP / (TP + FN); for multi-class we macro-average per-class recall.
- **Specificity**: TN / (TN + FP); for each class we treat it as the positive class and compute specificity, then macro-average.

In [ ]:
def accuracy(logits, labels):
    _, preds = torch.max(logits, dim=1)
    return (preds == labels).float().mean()

def compute_metrics(labels_np, preds_np, num_classes):
    """From numpy labels and preds, compute accuracy, F1 macro, sensitivity (recall) macro, specificity macro."""
    acc = (labels_np == preds_np).mean()
    f1_macro = f1_score(labels_np, preds_np, average="macro", zero_division=0)
    sensitivity_macro = recall_score(labels_np, preds_np, average="macro", zero_division=0)
    cm = confusion_matrix(labels_np, preds_np, labels=list(range(num_classes)))
    specificities = []
    for c in range(num_classes):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)
    specificity_macro = np.mean(specificities) if specificities else 0.0
    return {"accuracy": float(acc), "f1_macro": float(f1_macro), "sensitivity_macro": float(sensitivity_macro), "specificity_macro": float(specificity_macro)}

### Model: AlexNet

We use `torchvision.models.alexnet` and replace the classifier's last linear layer so the output has `num_classes` (10 for CIFAR-10). We can start from pretrained ImageNet weights or random init.

In [ ]:
# Use weights=None for random init (faster for demo); set weights=torchvision.models.AlexNet_Weights.IMAGENET1K_V1 for pretrained
model = torchvision.models.alexnet(weights=None)
model.classifier[-1] = nn.Linear(4096, num_classes)
print(model)

### Training loop and evaluation

We use the same pattern as the MNIST notebook: a wrapper provides `training_step`, `validation_step`, and `validation_epoch_end`. In validation we collect all predictions and labels so we can compute F1, sensitivity, and specificity from the confusion matrix.

In [ ]:
import torch.nn.functional as F

class AlexNetWrapper(nn.Module):
    """Thin wrapper so we can use training_step / validation_step / validation_epoch_end."""
    def __init__(self, model, num_classes=10):
        super().__init__()
        self.model = model
        self.num_classes = num_classes

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch):
        images, targets = batch
        out = self(images)
        return F.cross_entropy(out, targets)

    def validation_step(self, batch):
        images, targets = batch
        out = self(images)
        loss = F.cross_entropy(out, targets)
        _, preds = torch.max(out, dim=1)
        acc = (preds == targets).float().mean()
        return {"val_loss": loss.detach(), "val_acc": acc, "preds": preds.cpu().numpy(), "labels": targets.cpu().numpy()}

    def validation_epoch_end(self, outputs):
        batch_losses = [x["val_loss"] for x in outputs]
        batch_acc = [x["val_acc"] for x in outputs]
        all_preds = np.concatenate([x["preds"] for x in outputs])
        all_labels = np.concatenate([x["labels"] for x in outputs])
        metrics = compute_metrics(all_labels, all_preds, self.num_classes)
        return {
            "val_loss": torch.stack(batch_losses).mean().item(),
            "val_acc": torch.stack(batch_acc).mean().item(),
            "val_f1": metrics["f1_macro"],
            "val_sensitivity": metrics["sensitivity_macro"],
            "val_specificity": metrics["specificity_macro"],
        }

    def epoch_end(self, epoch, result, lr):
        print("Epoch [{}] - LR {}, train_loss: {:.4f}, val_loss: {:.4f}, val_acc: {:.4f}, val_f1: {:.4f}, val_sens: {:.4f}, val_spec: {:.4f}".format(
            epoch, lr, result["train_loss"], result["val_loss"], result["val_acc"],
            result["val_f1"], result["val_sensitivity"], result["val_specificity"]))

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

@torch.no_grad()
def evaluate(model, val_loader):
    model.eval()
    outputs = [model.validation_step(batch) for batch in val_loader]
    return model.validation_epoch_end(outputs)

def fit(epochs, lr, model, train_loader, val_loader, opt_func=torch.optim.Adam, writer=None):
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    history = []
    optimizer = opt_func(model.parameters(), lr=lr)
    scheduler = OneCycleLR(optimizer, lr, epochs=epochs, steps_per_epoch=len(train_loader))

    if writer is not None:
        writer.start_run(name="alexnet-base-v4", tags={"source": "notebook", "dataset": "cifar10", "method": "ptq"})
        writer.log_parameter(key="learning_rate", value=lr)
        writer.log_parameter(key="epochs", value=epochs)
        if hasattr(train_loader, "batch_size"):
            writer.log_parameter(key="batch_size", value=train_loader.batch_size)
        writer.emit_run_record()

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for batch in train_loader:
            loss = model.training_step(batch)
            train_losses.append(loss)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
        result = evaluate(model, val_loader)
        result["train_loss"] = torch.stack(train_losses).mean().item()
        model.epoch_end(epoch, result, scheduler.get_last_lr())
        history.append(result)
        if writer is not None:
            writer.log_metric(name="Training Loss", value=result["train_loss"], step=epoch)
            writer.log_metric(name="Validation Loss", value=result["val_loss"], step=epoch)
            writer.log_metric(name="Validation Accuracy", value=result["val_acc"], step=epoch)
            writer.log_metric(name="Validation F1 Score", value=result["val_f1"], step=epoch)
            writer.log_metric(name="Validation Sensitivity", value=result["val_sensitivity"], step=epoch)
            writer.log_metric(name="Validation Specificity", value=result["val_specificity"], step=epoch)
            writer.emit_run_record()
    if writer is not None:
        scalars = measure_run_scalars(model, device, val_loader)
        log_scalars_to_writer(writer, scalars)
        writer.log_summary_metrics({
            "train_loss": result["train_loss"], "val_loss": result["val_loss"], "val_acc": result["val_acc"],
            "val_f1": result["val_f1"], "val_sensitivity": result["val_sensitivity"], "val_specificity": result["val_specificity"],
        })
        writer.end_run(status="succeeded")
        writer.emit_run_record()
    return history

In [ ]:
def get_default_device():
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

def to_device(data, device):
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader:
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device
        self.batch_size = getattr(dl, "batch_size", None)

    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return len(self.dl)

device = get_default_device()
train_loader = DeviceDataLoader(train_loader, device)
val_loader = DeviceDataLoader(val_loader, device)
model = AlexNetWrapper(model, num_classes=num_classes)
model = model.to(device)

In [ ]:
history = [evaluate(model, val_loader)]
history

In [ ]:
# Helper functions
def get_model_size_mb(model: nn.Module) -> float:
    """Approximate model size in MB.

    - Regular FP models: sum parameter + buffer bytes.
    - Quantized/PT2E models: try to serialize; if that fails, fall back to
      an INT8 approximation for parameters plus real buffer sizes.
    """
    is_quantized = False
    is_pt2e_quantized = getattr(model, "_is_pt2e_quantized", False)

    # Heuristic: look for quantized modules / packed params
    try:
        for module in model.modules():
            module_type = str(type(module)).lower()
            if "quantized" in module_type or hasattr(module, "_packed_params"):
                is_quantized = True
                break
    except Exception:
        pass

    if is_pt2e_quantized:
        is_quantized = True

    if is_quantized:
        # PT2E quantized: approximate INT8 params + FP32-ish buffers
        if is_pt2e_quantized:
            param_size = 0
            buffer_size = 0
            for p in model.parameters():
                # Assume INT8 weights (1 byte per element)
                param_size += p.nelement() * 1
            for b in model.buffers():
                buffer_size += b.nelement() * b.element_size()
            return (param_size + buffer_size) / (1024 * 1024)

        # Legacy quantized: try serialization first
        try:
            buf = io.BytesIO()
            torch.save(model, buf)
            return buf.tell() / (1024 * 1024)
        except Exception:
            # Fallback: INT8 approximation
            param_size = 0
            buffer_size = 0
            for p in model.parameters():
                param_size += p.nelement() * 1
            for b in model.buffers():
                buffer_size += b.nelement() * b.element_size()
            return (param_size + buffer_size) / (1024 * 1024)

    # Regular FP32/FP16 model
    param_size = 0
    buffer_size = 0
    for p in model.parameters():
        param_size += p.nelement() * p.element_size()
    for b in model.buffers():
        buffer_size += b.nelement() * b.element_size()
    return (param_size + buffer_size) / (1024 * 1024)

def measure_run_scalars(model, device, data_loader):
    """Simple profile: model_size_mb, latency_ms, throughput_ips. Table-only."""
    model.eval()
    m = getattr(model, "model", model)
    scalars = {}
    scalars["model_size_mb"] = get_model_size_mb(model)
    if device.type == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        scalars["memory_usage_mb"] = torch.cuda.max_memory_allocated(device) / 1e6
    try:
        batch = next(iter(data_loader))
        inp = batch[0].to(device) if isinstance(batch, (list, tuple)) else batch.to(device)
        batch_size = inp.shape[0]
        with torch.no_grad():
            for _ in range(2):
                model(inp)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times = []
            for _ in range(15):
                t0 = time.perf_counter()
                model(inp)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000.0)
        lat_ms = sorted(times)[len(times) // 2]
        scalars["latency_p50_ms"] = lat_ms
        if batch_size and lat_ms > 0:
            scalars["throughput_ips"] = batch_size * 1000.0 / lat_ms
    except Exception:
        pass
    return scalars

def log_scalars_to_writer(writer, scalars):
    units = {"model_size_mb": "MB", "memory_usage_mb": "MB", "latency_p50_ms": "ms", "throughput_ips": "samples/s"}
    for k, v in scalars.items():
        writer.log_scalar(k, v, unit=units.get(k, ""))

import copy
from torch.ao.quantization import fuse_modules, get_default_qconfig, prepare, convert
from torch.quantization import QuantStub, DeQuantStub


def ensure_quantized_backend(backend: str = "fbgemm") -> str:
    """Set and return a valid quantized backend for the current CPU.

    Prefers the requested backend when available, otherwise falls back to a
    supported engine (fbgemm or qnnpack).
    """
    try:
        supported = torch.backends.quantized.supported_engines
    except AttributeError:
        torch.backends.quantized.engine = backend
        return backend

    if backend in supported:
        torch.backends.quantized.engine = backend
        return backend

    if "fbgemm" in supported:
        torch.backends.quantized.engine = "fbgemm"
        return "fbgemm"
    if "qnnpack" in supported:
        torch.backends.quantized.engine = "qnnpack"
        return "qnnpack"

    fallback = supported[0] if supported else backend
    torch.backends.quantized.engine = fallback
    return fallback


def _fuse_alexnet_ptq(model: nn.Module) -> None:
    """Fuse Conv+ReLU pairs in torchvision AlexNet-style models for PTQ."""
    try:
        features = getattr(model, "features", None)
        if isinstance(features, nn.Sequential):
            i = 0
            while i < len(features) - 1:
                if isinstance(features[i], nn.Conv2d) and isinstance(features[i + 1], (nn.ReLU, nn.ReLU6)):
                    try:
                        fuse_modules(features, [str(i), str(i + 1)], inplace=True)
                    except Exception:
                        pass
                    i += 2
                else:
                    i += 1
    except Exception as e:
        print(f"Warning: AlexNet fusion encountered an error: {e}")


def prepare_model_for_quantization(model: nn.Module) -> nn.Module:
    """Attach QuantStub/DeQuantStub if missing and set eval mode."""
    model.eval()
    if not hasattr(model, "quant"):
        model.quant = QuantStub()
    if not hasattr(model, "dequant"):
        model.dequant = DeQuantStub()
    return model


def quantize_model_static_ptq(model: nn.Module, calibration_loader, model_name: str, backend: str = "fbgemm") -> nn.Module:
    """Static PTQ with module fusion: fuse → prepare → calibrate → convert.

    Mirrors the compression-engine static PTQ flow for AlexNet and similar
    conv nets.
    """
    print(f"Starting static PTQ with fusion for {model_name}...")

    # Work on a CPU copy
    model_cpu = copy.deepcopy(model).cpu().eval()

    backend_actual = ensure_quantized_backend(backend)
    print(f"Using quantization backend: {backend_actual}")

    # Architecture-specific fusion (AlexNet)
    if model_name.lower() == "alexnet":
        print("Fusing Conv+ReLU modules for AlexNet...")
        _fuse_alexnet_ptq(model_cpu)

    # Add QuantStub/DeQuantStub and set qconfig
    model_cpu = prepare_model_for_quantization(model_cpu)
    model_cpu.qconfig = get_default_qconfig(backend_actual)

    # Insert observers
    print("Preparing model with observers for calibration...")
    model_cpu = prepare(model_cpu, inplace=False)

    # Calibrate on representative data
    print("Calibrating model on representative data...")
    model_cpu.eval()
    with torch.no_grad():
        num_batches = min(100, len(calibration_loader))
        for i, batch in enumerate(calibration_loader):
            if i >= num_batches:
                break
            data = batch[0] if isinstance(batch, (tuple, list)) else batch
            model_cpu(data.cpu())

    # Convert to quantized model
    print("Converting to INT8 quantized model...")
    quantized_model = convert(model_cpu, inplace=False)
    ensure_quantized_backend(backend_actual)
    print("Static PTQ with fusion successful!\n")
    return quantized_model

In [ ]:
# PTQ (updated): use compression-engine style static PTQ with fusion.
backend = "fbgemm"  # use "qnnpack" on ARM if desired

# Get the raw AlexNet from the wrapper and put in eval mode
model_fp32 = model.model.eval().cpu()

# Calibration loader on CPU (no DeviceDataLoader here; quantization runs on CPU)
calib_loader = DataLoader(val_ds, batch_size=64, num_workers=0)

# Quantize using static PTQ with fusion (helpers defined earlier in this notebook)
model_q = quantize_model_static_ptq(
    model_fp32,
    calib_loader,
    model_name="alexnet",
    backend=backend,
)

In [ ]:
# CEMI: optional setup for experiment logging. Run this cell to log the next training run.
# Then run `cemi gateway` and `cemi view` (from repo root) to see runs in the workspace.
# Use the repo root .cemi so the gateway (started from repo root) sees these runs.
from cemi.writer import create_writer

log_dir = "/media/volume/volume-dev/projects/cemi/.cemi"
writer = create_writer(project="alexnet-ptq", log_dir=log_dir)

In [ ]:
# Run the CEMI setup cell above to log this run to CEMI.
history += fit(100, 1e-3, model, train_loader, val_loader, writer=writer if "writer" in dir() else None)

### PTQ: what the config does

PyTorch's quantization **config** (e.g. `get_default_qconfig_mapping()`) specifies *what* to quantize (weights and activations) and *how* (e.g. symmetric INT8, per-tensor or per-channel). The **backend** (e.g. `"qnnpack"` for ARM, `"fbgemm"` for x86) decides the exact low-level ops. **prepare_fx** inserts observers into the graph; during **calibration** we run the model on representative data so those observers record min/max (or histogram) and compute scale and zero-point. **convert_fx** then replaces float modules with quantized integer ones using those ranges.

In [ ]:
# PTQ: prepare and calibrate. We use the inner AlexNet (not the wrapper) for quantization.
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
backend = "fbgemm"  # use "qnnpack" for ARM
try:
    from torch.ao.quantization import get_default_backend_config_dict
    backend_config = get_default_backend_config_dict(backend)
except Exception:
    backend_config = None

# Get the raw AlexNet from the wrapper and put in eval mode
model_fp32 = model.model.eval().cpu()
# Example input for FX tracing (batch of 1)
example_input = next(iter(DeviceDataLoader(DataLoader(val_ds, batch_size=1), torch.device("cpu"))))[0]
qconfig_mapping = get_default_qconfig_mapping(backend)

# Prepare: insert observers (backend_config optional in some PyTorch versions)
kwargs = {"backend_config": backend_config} if backend_config is not None else {}
model_prepared = prepare_fx(model_fp32, qconfig_mapping, example_input, **kwargs)
# Calibrate: run a few batches so observers record ranges
model_prepared.eval()
calib_loader = DeviceDataLoader(DataLoader(val_ds, batch_size=64, num_workers=0), torch.device("cpu"))
for i, (images, _) in enumerate(calib_loader):
    if i >= 4:
        break
    model_prepared(images)
# Convert to quantized model
model_q = convert_fx(model_prepared, qconfig_mapping=qconfig_mapping, **kwargs)

In [ ]:
# Evaluate quantized model (runs on CPU with fbgemm). Wrap in AlexNetWrapper so evaluate() works.
model_q_wrapper = AlexNetWrapper(model_q, num_classes=num_classes)
val_loader_cpu = DeviceDataLoader(DataLoader(val_ds, batch_size=batch_size * 2, num_workers=0), torch.device("cpu"))
result_q = evaluate(model_q_wrapper, val_loader_cpu)
print("Quantized model - val_loss: {:.4f}, val_acc: {:.4f}, val_f1: {:.4f}, val_sensitivity: {:.4f}, val_specificity: {:.4f}".format(
    result_q["val_loss"], result_q["val_acc"], result_q["val_f1"], result_q["val_sensitivity"], result_q["val_specificity"]))

# Log quantized run to CEMI as a second run (optional)
if "writer" in dir() and writer is not None:
    writer.start_run(name="alexnet-ptq-quantized-v4", tags={"source": "notebook", "dataset": "cifar10", "method": "ptq"})
    writer.log_parameter(key="quantized", value=True)
    scalars_q = measure_run_scalars(model_q_wrapper, torch.device("cpu"), val_loader_cpu)
    log_scalars_to_writer(writer, scalars_q)
    writer.log_summary_metrics({
        "val_loss": result_q["val_loss"], "val_acc": result_q["val_acc"],
        "val_f1": result_q["val_f1"], "val_sensitivity": result_q["val_sensitivity"], "val_specificity": result_q["val_specificity"],
    })
    writer.end_run(status="succeeded")
    writer.emit_run_record()

### Plot metrics

Plot training and validation loss and accuracy (and optionally F1, sensitivity, specificity) from `history`.

In [ ]:
def plot_losses(history):
    train_losses = [x.get("train_loss") for x in history]
    val_losses = [x["val_loss"] for x in history]
    plt.plot(train_losses, "-bx", label="Train loss")
    plt.plot(val_losses, "-rx", label="Val loss")
    plt.xlabel("epoch")
    plt.legend()
    plt.title("Loss vs. epochs")
    plt.show()

def plot_scores(history):
    accs = [x["val_acc"] for x in history]
    f1s = [x["val_f1"] for x in history]
    sens = [x["val_sensitivity"] for x in history]
    spec = [x["val_specificity"] for x in history]
    plt.plot(accs, "-bx", label="Val accuracy")
    plt.plot(f1s, "-gx", label="Val F1 (macro)")
    plt.plot(sens, "-cx", label="Val sensitivity")
    plt.plot(spec, "-mx", label="Val specificity")
    plt.xlabel("epoch")
    plt.legend()
    plt.title("Metrics vs. epochs")
    plt.show()

plot_losses(history)
plot_scores(history)

### Conclusion

We trained AlexNet (FP32) on CIFAR-10, applied post-training quantization with PyTorch's FX path (prepare_fx → calibrate → convert_fx), and compared validation metrics. All runs and metrics were logged to CEMI so you can inspect them in the workspace via `cemi gateway` and `cemi view`.